# HPRC Ensembl vs CAT Annotation QC Report

This notebook generates a comprehensive QC report comparing Ensembl (linear projection) and CAT (graph-based projection) gene annotations across HPRC assemblies.

## Data Sources
- Transcript concordance metrics
- Coding sequence integrity assessments
- Gene presence/absence comparisons
- Multi-mapping patterns
- Overall overlap statistics (RBH pairs)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 1. Load Data

Load all QC metric files from the pipeline output directory.

In [ ]:
# Set path to pipeline output directory
OUTPUT_DIR = Path('../results')  # Adjust this path as needed
QC_DIR = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'

print(f"Loading data from: {OUTPUT_DIR}")
print(f"QC metrics directory: {QC_DIR}")
print(f"Results directory: {RESULTS_DIR}")

In [ ]:
def load_all_tsvs(pattern, base_dir):
    """Load and concatenate all TSV files matching pattern."""
    files = list(base_dir.rglob(pattern))
    print(f"Found {len(files)} files matching {pattern}")
    
    if not files:
        print(f"WARNING: No files found for pattern {pattern}")
        return pd.DataFrame()
    
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, sep='\t')
            dfs.append(df)
        except Exception as e:
            print(f"Error loading {f}: {e}")
    
    if dfs:
        combined = pd.concat(dfs, ignore_index=True)
        print(f"Loaded {len(combined)} total rows")
        return combined
    return pd.DataFrame()

# Load each QC metric type
print("\n=== Loading Transcript Concordance ===")
transcript_conc = load_all_tsvs('*_transcript_concordance.tsv', QC_DIR)

print("\n=== Loading Coding Integrity ===")
coding_int = load_all_tsvs('*_coding_integrity.tsv', QC_DIR)

print("\n=== Loading Gene Presence/Absence ===")
gene_presence = load_all_tsvs('*_gene_presence.tsv', QC_DIR)

print("\n=== Loading Multi-Mapping ===")
multi_mapping = load_all_tsvs('*_multi_mapping.tsv', QC_DIR)

print("\n=== Loading RBH Pairs ===")
rbh_pairs = load_all_tsvs('*.gene_pairs_rbh.tsv', RESULTS_DIR)

print("\n=== Data Loading Complete ===")

In [ ]:
# Display sample data and columns
print("Transcript Concordance columns:", list(transcript_conc.columns) if not transcript_conc.empty else "No data")
print("Coding Integrity columns:", list(coding_int.columns) if not coding_int.empty else "No data")
print("Gene Presence columns:", list(gene_presence.columns) if not gene_presence.empty else "No data")
print("Multi-Mapping columns:", list(multi_mapping.columns) if not multi_mapping.empty else "No data")
print("RBH Pairs columns:", list(rbh_pairs.columns) if not rbh_pairs.empty else "No data")

## 2. Overall Summary Statistics

In [ ]:
# Number of assemblies analyzed
n_assemblies = transcript_conc['assembly_accession'].nunique() if not transcript_conc.empty else 0
print(f"\n{'='*60}")
print(f"HPRC ANNOTATION QC SUMMARY")
print(f"{'='*60}")
print(f"Total assemblies analyzed: {n_assemblies}")
print(f"Total RBH gene pairs: {len(rbh_pairs):,}" if not rbh_pairs.empty else "No RBH data")
print(f"Total transcript comparisons: {len(transcript_conc):,}" if not transcript_conc.empty else "No transcript data")
print(f"Total protein-coding genes assessed: {len(coding_int):,}" if not coding_int.empty else "No coding data")
print(f"{'='*60}\n")

## 3. Transcript Concordance Analysis

For RBH gene pairs, what proportion have exact transcript structure matches?

In [ ]:
if not transcript_conc.empty:
    # Per-assembly summary
    assembly_tx_summary = transcript_conc.groupby('assembly_accession').agg({
        'transcript_concordance_rate': 'mean',
        'n_exact_matches': 'sum',
        'n_partial_matches': 'sum',
        'ensembl_gene_id': 'count'
    }).round(4)
    assembly_tx_summary.columns = ['mean_concordance_rate', 'total_exact_matches', 
                                     'total_partial_matches', 'n_genes']
    
    print("Per-Assembly Transcript Concordance Summary:")
    print(assembly_tx_summary.describe())
    
    # Overall statistics
    overall_concordance = transcript_conc['transcript_concordance_rate'].mean()
    genes_with_perfect = (transcript_conc['transcript_concordance_rate'] == 1.0).sum()
    genes_with_some = (transcript_conc['n_exact_matches'] > 0).sum()
    
    print(f"\n--- Overall Transcript Concordance ---")
    print(f"Mean concordance rate across all genes: {overall_concordance:.2%}")
    print(f"Genes with 100% transcript concordance: {genes_with_perfect:,} ({genes_with_perfect/len(transcript_conc):.1%})")
    print(f"Genes with at least one exact match: {genes_with_some:,} ({genes_with_some/len(transcript_conc):.1%})")
else:
    print("No transcript concordance data available")

In [ ]:
if not transcript_conc.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Plot 1: Distribution of concordance rates
    axes[0].hist(transcript_conc['transcript_concordance_rate'], bins=50, edgecolor='black')
    axes[0].axvline(overall_concordance, color='red', linestyle='--', 
                    label=f'Mean: {overall_concordance:.2%}')
    axes[0].set_xlabel('Transcript Concordance Rate')
    axes[0].set_ylabel('Number of Genes')
    axes[0].set_title('Distribution of Transcript Concordance\n(Per RBH Gene Pair)')
    axes[0].legend()
    
    # Plot 2: Exact vs partial matches
    match_counts = transcript_conc[['n_exact_matches', 'n_partial_matches']].sum()
    axes[1].bar(['Exact Matches', 'Partial Matches'], match_counts.values, 
                color=['green', 'orange'], edgecolor='black')
    axes[1].set_ylabel('Total Count Across All Genes')
    axes[1].set_title('Exact vs Partial Transcript Matches')
    for i, v in enumerate(match_counts.values):
        axes[1].text(i, v, f'{int(v):,}', ha='center', va='bottom')
    
    # Plot 3: Per-assembly mean concordance
    assembly_means = transcript_conc.groupby('assembly_accession')['transcript_concordance_rate'].mean().sort_values()
    axes[2].hist(assembly_means, bins=30, edgecolor='black')
    axes[2].axvline(assembly_means.mean(), color='red', linestyle='--', 
                    label=f'Mean: {assembly_means.mean():.2%}')
    axes[2].set_xlabel('Mean Concordance Rate')
    axes[2].set_ylabel('Number of Assemblies')
    axes[2].set_title('Per-Assembly Mean Transcript Concordance')
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'transcript_concordance_summary.png', dpi=300, bbox_inches='tight')
    plt.show()

## 4. Coding Sequence Integrity

For protein-coding genes, assess start/stop codon agreement and frameshift detection.

In [ ]:
if not coding_int.empty:
    # Convert boolean strings to actual booleans
    for col in ['start_codon_match', 'stop_codon_match', 'frameshift_detected', 
                'has_ensembl_cds', 'has_cat_cds']:
        if col in coding_int.columns:
            coding_int[col] = coding_int[col].map({'True': True, 'False': False, True: True, False: False})
    
    # Filter to genes with CDS in both annotations
    with_cds = coding_int[
        (coding_int['has_ensembl_cds'] == True) & 
        (coding_int['has_cat_cds'] == True)
    ]
    
    print(f"\n--- Coding Sequence Integrity ---")
    print(f"Total protein-coding RBH pairs: {len(coding_int):,}")
    print(f"Pairs with CDS in both annotations: {len(with_cds):,} ({len(with_cds)/len(coding_int):.1%})")
    
    if len(with_cds) > 0:
        start_match = (with_cds['start_codon_match'] == True).sum()
        stop_match = (with_cds['stop_codon_match'] == True).sum()
        both_match = ((with_cds['start_codon_match'] == True) & 
                     (with_cds['stop_codon_match'] == True)).sum()
        frameshifts = (with_cds['frameshift_detected'] == True).sum()
        
        print(f"\nStart codon matches: {start_match:,} ({start_match/len(with_cds):.1%})")
        print(f"Stop codon matches: {stop_match:,} ({stop_match/len(with_cds):.1%})")
        print(f"Both start AND stop match: {both_match:,} ({both_match/len(with_cds):.1%})")
        print(f"Potential frameshifts detected: {frameshifts:,} ({frameshifts/len(with_cds):.1%})")
        
        # CDS length differences
        print(f"\nCDS Length Statistics:")
        print(f"Mean length difference: {with_cds['length_difference'].mean():.1f} bp")
        print(f"Median length difference: {with_cds['length_difference'].median():.1f} bp")
        print(f"Genes with identical CDS length: {(with_cds['length_difference'] == 0).sum():,} ({(with_cds['length_difference'] == 0).sum()/len(with_cds):.1%})")
else:
    print("No coding integrity data available")
    with_cds = pd.DataFrame()

In [ ]:
if not with_cds.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Start/Stop codon agreement
    categories = ['Start\nMatch', 'Stop\nMatch', 'Both\nMatch', 'Frameshift']
    counts = [
        (with_cds['start_codon_match'] == True).sum(),
        (with_cds['stop_codon_match'] == True).sum(),
        ((with_cds['start_codon_match'] == True) & (with_cds['stop_codon_match'] == True)).sum(),
        (with_cds['frameshift_detected'] == True).sum()
    ]
    percentages = [c/len(with_cds)*100 for c in counts]
    colors = ['green', 'green', 'darkgreen', 'red']
    
    axes[0,0].bar(categories, percentages, color=colors, edgecolor='black', alpha=0.7)
    axes[0,0].set_ylabel('Percentage of Genes')
    axes[0,0].set_title('Codon Position Agreement\n(Protein-Coding RBH Pairs)')
    axes[0,0].set_ylim([0, 105])
    for i, (c, p) in enumerate(zip(counts, percentages)):
        axes[0,0].text(i, p+2, f'{p:.1f}%\n({c:,})', ha='center', va='bottom')
    
    # Plot 2: CDS length difference distribution
    axes[0,1].hist(with_cds['length_difference'], bins=50, edgecolor='black')
    axes[0,1].set_xlabel('Absolute CDS Length Difference (bp)')
    axes[0,1].set_ylabel('Number of Genes')
    axes[0,1].set_title('Distribution of CDS Length Differences')
    axes[0,1].axvline(0, color='red', linestyle='--', label='Perfect match')
    axes[0,1].legend()
    
    # Plot 3: Length difference (zoomed to small differences)
    small_diffs = with_cds[with_cds['length_difference'] <= 100]
    axes[1,0].hist(small_diffs['length_difference'], bins=50, edgecolor='black')
    axes[1,0].set_xlabel('CDS Length Difference (bp)')
    axes[1,0].set_ylabel('Number of Genes')
    axes[1,0].set_title(f'CDS Length Differences ≤100 bp\n({len(small_diffs):,} genes)')
    
    # Plot 4: Scatter - Ensembl vs CAT CDS length
    sample = with_cds.sample(min(5000, len(with_cds)))
    axes[1,1].scatter(sample['cds_length_ensembl'], sample['cds_length_cat'], 
                     alpha=0.3, s=10)
    max_len = max(sample['cds_length_ensembl'].max(), sample['cds_length_cat'].max())
    axes[1,1].plot([0, max_len], [0, max_len], 'r--', alpha=0.5, label='Perfect agreement')
    axes[1,1].set_xlabel('Ensembl CDS Length (bp)')
    axes[1,1].set_ylabel('CAT CDS Length (bp)')
    axes[1,1].set_title(f'CDS Length Comparison\n(Sample of {len(sample):,} genes)')
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'coding_integrity_summary.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Gene Presence/Absence Analysis

Which genes are present in both annotations vs unique to one?

In [ ]:
if not gene_presence.empty:
    # Convert boolean strings
    gene_presence['present_in_ensembl'] = gene_presence['present_in_ensembl'].map(
        {'True': True, 'False': False, True: True, False: False}
    )
    gene_presence['present_in_cat'] = gene_presence['present_in_cat'].map(
        {'True': True, 'False': False, True: True, False: False}
    )
    
    # Classify genes
    both = gene_presence[
        (gene_presence['present_in_ensembl'] == True) & 
        (gene_presence['present_in_cat'] == True)
    ]
    ensembl_only = gene_presence[
        (gene_presence['present_in_ensembl'] == True) & 
        (gene_presence['present_in_cat'] == False)
    ]
    cat_only = gene_presence[
        (gene_presence['present_in_ensembl'] == False) & 
        (gene_presence['present_in_cat'] == True)
    ]
    
    print(f"\n--- Gene Presence/Absence by Name ---")
    print(f"Total unique named genes: {gene_presence['gene_name'].nunique():,}")
    print(f"\nPresent in BOTH: {len(both):,} ({len(both)/len(gene_presence):.1%})")
    print(f"Ensembl ONLY: {len(ensembl_only):,} ({len(ensembl_only)/len(gene_presence):.1%})")
    print(f"CAT ONLY: {len(cat_only):,} ({len(cat_only)/len(gene_presence):.1%})")
    
    # Top missing genes
    if len(ensembl_only) > 0:
        print(f"\nTop genes in Ensembl but NOT in CAT:")
        top_ens = ensembl_only.groupby('gene_name').size().sort_values(ascending=False).head(10)
        for gene, count in top_ens.items():
            print(f"  {gene}: {count} assemblies")
    
    if len(cat_only) > 0:
        print(f"\nTop genes in CAT but NOT in Ensembl:")
        top_cat = cat_only.groupby('gene_name').size().sort_values(ascending=False).head(10)
        for gene, count in top_cat.items():
            print(f"  {gene}: {count} assemblies")
else:
    print("No gene presence data available")
    both = ensembl_only = cat_only = pd.DataFrame()

In [ ]:
if not gene_presence.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Overall presence/absence
    categories = ['Both\nAnnotations', 'Ensembl\nOnly', 'CAT\nOnly']
    counts = [len(both), len(ensembl_only), len(cat_only)]
    colors = ['green', 'blue', 'orange']
    
    axes[0].bar(categories, counts, color=colors, edgecolor='black', alpha=0.7)
    axes[0].set_ylabel('Number of Gene Occurrences\n(across all assemblies)')
    axes[0].set_title('Gene Presence/Absence by Name')
    for i, c in enumerate(counts):
        pct = c/len(gene_presence)*100
        axes[0].text(i, c, f'{c:,}\n({pct:.1f}%)', ha='center', va='bottom')
    
    # Plot 2: Per-assembly consistency
    assembly_stats = gene_presence.groupby('assembly_accession').apply(
        lambda x: pd.Series({
            'both': ((x['present_in_ensembl'] == True) & (x['present_in_cat'] == True)).sum(),
            'ens_only': ((x['present_in_ensembl'] == True) & (x['present_in_cat'] == False)).sum(),
            'cat_only': ((x['present_in_ensembl'] == False) & (x['present_in_cat'] == True)).sum()
        })
    )
    
    axes[1].hist([assembly_stats['ens_only'], assembly_stats['cat_only']], 
                 bins=30, label=['Ensembl-only genes', 'CAT-only genes'],
                 color=['blue', 'orange'], alpha=0.6, edgecolor='black')
    axes[1].set_xlabel('Number of Unique Genes per Assembly')
    axes[1].set_ylabel('Number of Assemblies')
    axes[1].set_title('Distribution of Annotation-Specific Genes')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'gene_presence_summary.png', dpi=300, bbox_inches='tight')
    plt.show()

## 6. Multi-Mapping Analysis

Identify genes with complex 1-to-many or many-to-1 relationships.

In [ ]:
if not multi_mapping.empty:
    # Separate by source
    ensembl_multi = multi_mapping[multi_mapping['source'] == 'ensembl']
    cat_multi = multi_mapping[multi_mapping['source'] == 'cat']
    
    print(f"\n--- Multi-Mapping Analysis ---")
    print(f"Ensembl genes with multiple CAT matches: {len(ensembl_multi):,}")
    print(f"CAT genes with multiple Ensembl matches: {len(cat_multi):,}")
    
    if len(ensembl_multi) > 0:
        print(f"\nEnsembl 1-to-many statistics:")
        print(f"  Mean matches per gene: {ensembl_multi['n_matches'].mean():.2f}")
        print(f"  Max matches: {ensembl_multi['n_matches'].max()}")
        print(f"  Genes with 2 matches: {(ensembl_multi['n_matches'] == 2).sum():,}")
        print(f"  Genes with 3+ matches: {(ensembl_multi['n_matches'] >= 3).sum():,}")
    
    if len(cat_multi) > 0:
        print(f"\nCAT many-to-1 statistics:")
        print(f"  Mean matches per gene: {cat_multi['n_matches'].mean():.2f}")
        print(f"  Max matches: {cat_multi['n_matches'].max()}")
        print(f"  Genes with 2 matches: {(cat_multi['n_matches'] == 2).sum():,}")
        print(f"  Genes with 3+ matches: {(cat_multi['n_matches'] >= 3).sum():,}")
else:
    print("No multi-mapping data available")
    ensembl_multi = cat_multi = pd.DataFrame()

In [ ]:
if not multi_mapping.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Distribution of match counts
    if len(ensembl_multi) > 0:
        axes[0].hist(ensembl_multi['n_matches'], bins=range(2, ensembl_multi['n_matches'].max()+2),
                     edgecolor='black', alpha=0.7, color='blue')
        axes[0].set_xlabel('Number of CAT Gene Matches')
        axes[0].set_ylabel('Number of Ensembl Genes')
        axes[0].set_title('1-to-Many: Ensembl Genes with Multiple CAT Matches')
        axes[0].set_xticks(range(2, min(11, ensembl_multi['n_matches'].max()+1)))
    
    # Plot 2: CAT multi-mapping
    if len(cat_multi) > 0:
        axes[1].hist(cat_multi['n_matches'], bins=range(2, cat_multi['n_matches'].max()+2),
                     edgecolor='black', alpha=0.7, color='orange')
        axes[1].set_xlabel('Number of Ensembl Gene Matches')
        axes[1].set_ylabel('Number of CAT Genes')
        axes[1].set_title('Many-to-1: CAT Genes with Multiple Ensembl Matches')
        axes[1].set_xticks(range(2, min(11, cat_multi['n_matches'].max()+1)))
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'multi_mapping_summary.png', dpi=300, bbox_inches='tight')
    plt.show()

## 7. RBH Overlap Quality

For reciprocal best hit gene pairs, assess the quality of spatial overlap.

In [ ]:
if not rbh_pairs.empty:
    print(f"\n--- RBH Pair Overlap Quality ---")
    print(f"Total RBH pairs: {len(rbh_pairs):,}")
    
    # Overlap statistics
    high_overlap = ((rbh_pairs['frac_ensembl_covered'] >= 0.9) & 
                    (rbh_pairs['frac_cat_covered'] >= 0.9)).sum()
    
    print(f"\nPairs with ≥90% reciprocal overlap: {high_overlap:,} ({high_overlap/len(rbh_pairs):.1%})")
    print(f"Mean Ensembl coverage: {rbh_pairs['frac_ensembl_covered'].mean():.1%}")
    print(f"Mean CAT coverage: {rbh_pairs['frac_cat_covered'].mean():.1%}")
    
    # Classification breakdown
    if 'classification' in rbh_pairs.columns:
        print(f"\nOverlap Classification:")
        class_counts = rbh_pairs['classification'].value_counts()
        for cls, count in class_counts.items():
            print(f"  {cls}: {count:,} ({count/len(rbh_pairs):.1%})")
else:
    print("No RBH pairs data available")

In [ ]:
if not rbh_pairs.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Scatter of reciprocal coverage
    sample = rbh_pairs.sample(min(10000, len(rbh_pairs)))
    axes[0].scatter(sample['frac_ensembl_covered'], sample['frac_cat_covered'],
                   alpha=0.3, s=10)
    axes[0].axhline(0.9, color='red', linestyle='--', alpha=0.5, label='90% threshold')
    axes[0].axvline(0.9, color='red', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('Fraction of Ensembl Gene Covered')
    axes[0].set_ylabel('Fraction of CAT Gene Covered')
    axes[0].set_title(f'Reciprocal Coverage in RBH Pairs\n(Sample of {len(sample):,} pairs)')
    axes[0].legend()
    axes[0].set_xlim([0, 1.05])
    axes[0].set_ylim([0, 1.05])
    
    # Plot 2: Classification breakdown
    if 'classification' in rbh_pairs.columns:
        class_counts = rbh_pairs['classification'].value_counts()
        axes[1].barh(range(len(class_counts)), class_counts.values, edgecolor='black')
        axes[1].set_yticks(range(len(class_counts)))
        axes[1].set_yticklabels(class_counts.index)
        axes[1].set_xlabel('Number of Gene Pairs')
        axes[1].set_title('RBH Pair Overlap Classification')
        for i, (v, pct) in enumerate(zip(class_counts.values, 
                                         class_counts.values/len(rbh_pairs)*100)):
            axes[1].text(v, i, f' {v:,} ({pct:.1f}%)', va='center')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'rbh_overlap_quality.png', dpi=300, bbox_inches='tight')
    plt.show()

## 8. Summary Table

Generate a comprehensive summary table for the report.

In [ ]:
summary_data = []

if not transcript_conc.empty:
    summary_data.append([
        'Transcript Concordance',
        f"{transcript_conc['transcript_concordance_rate'].mean():.1%}",
        f"{(transcript_conc['transcript_concordance_rate'] == 1.0).sum():,} / {len(transcript_conc):,}",
        'Mean rate across RBH gene pairs'
    ])

if not with_cds.empty:
    both_match = ((with_cds['start_codon_match'] == True) & 
                  (with_cds['stop_codon_match'] == True)).sum()
    summary_data.append([
        'Start & Stop Codon Agreement',
        f"{both_match/len(with_cds):.1%}",
        f"{both_match:,} / {len(with_cds):,}",
        'Protein-coding genes with CDS'
    ])
    
    frameshifts = (with_cds['frameshift_detected'] == True).sum()
    summary_data.append([
        'Potential Frameshifts',
        f"{frameshifts/len(with_cds):.1%}",
        f"{frameshifts:,} / {len(with_cds):,}",
        'Non-divisible-by-3 length diffs'
    ])

if not gene_presence.empty:
    summary_data.append([
        'Genes in Both Annotations',
        f"{len(both)/len(gene_presence):.1%}",
        f"{len(both):,} / {len(gene_presence):,}",
        'Named genes present in both'
    ])

if not rbh_pairs.empty:
    high_overlap = ((rbh_pairs['frac_ensembl_covered'] >= 0.9) & 
                    (rbh_pairs['frac_cat_covered'] >= 0.9)).sum()
    summary_data.append([
        'High-Quality RBH Pairs',
        f"{high_overlap/len(rbh_pairs):.1%}",
        f"{high_overlap:,} / {len(rbh_pairs):,}",
        '≥90% reciprocal overlap'
    ])

summary_df = pd.DataFrame(summary_data, 
                         columns=['Metric', 'Percentage', 'Count', 'Description'])

print("\n" + "="*80)
print("OVERALL QC SUMMARY TABLE")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

# Save summary table
summary_df.to_csv(OUTPUT_DIR / 'qc_summary_table.tsv', sep='\t', index=False)
print(f"\nSummary table saved to: {OUTPUT_DIR / 'qc_summary_table.tsv'}")

## 9. Export Summary Statistics

Save detailed summary statistics for downstream analysis.

In [ ]:
# Per-assembly summary
if not transcript_conc.empty:
    per_assembly_summary = transcript_conc.groupby(['assembly_accession', 'sample_name']).agg({
        'transcript_concordance_rate': 'mean',
        'n_exact_matches': 'sum',
        'ensembl_gene_id': 'count'
    }).reset_index()
    per_assembly_summary.columns = ['assembly_accession', 'sample_name', 
                                     'mean_transcript_concordance', 
                                     'total_exact_transcript_matches',
                                     'n_rbh_genes_analyzed']
    
    # Add coding integrity stats if available
    if not coding_int.empty:
        coding_summary = coding_int.groupby(['assembly_accession', 'sample_name']).apply(
            lambda x: pd.Series({
                'start_stop_codon_agreement': 
                    ((x['start_codon_match'] == True) & (x['stop_codon_match'] == True)).sum() / len(x)
                    if len(x) > 0 else np.nan,
                'n_frameshifts': (x['frameshift_detected'] == True).sum()
            })
        ).reset_index()
        
        per_assembly_summary = per_assembly_summary.merge(
            coding_summary, on=['assembly_accession', 'sample_name'], how='left'
        )
    
    per_assembly_summary.to_csv(OUTPUT_DIR / 'per_assembly_qc_summary.tsv', 
                                 sep='\t', index=False)
    print(f"Per-assembly summary saved to: {OUTPUT_DIR / 'per_assembly_qc_summary.tsv'}")
    print(f"Total assemblies in summary: {len(per_assembly_summary)}")

## Conclusions

This report provides a comprehensive QC assessment of Ensembl (linear projection) vs CAT (graph-based projection) gene annotations across the HPRC assemblies.

### Key Findings:
1. **Transcript-level concordance** shows how well the two methods agree on alternative splicing patterns
2. **Coding sequence integrity** validates start/stop codons and detects potential annotation errors
3. **Gene presence/absence** identifies genes unique to each annotation method
4. **Multi-mapping patterns** highlight complex genomic regions with ambiguous annotations

### Next Steps:
- Investigate genes with low concordance or coding integrity issues
- Examine genes unique to one annotation method
- Analyze multi-mapping genes for paralogs or segmental duplications
- Compare results across ancestral populations